In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Section 4.3 — ML in Operations

**Objectives**

By the end of this section you will be able to:

- Build a demand forecasting model using engineered time-based features.
- Explain why time-series data must not be shuffled before splitting.
- Train a predictive maintenance classifier on machine sensor data.
- Articulate the business value of predictive versus reactive maintenance.

> **Cooking analogy:** A well-run kitchen never runs out of ingredients and
> never lets equipment break down during service. **Demand forecasting** tells
> the kitchen exactly how many covers to expect so it orders precisely the right
> quantities. **Predictive maintenance** listens to the sounds an oven makes and
> schedules servicing *before* it fails — just as an experienced chef notices
> when a flame is behaving strangely and calls the engineer before the dinner
> rush begins.

Operations teams use ML for demand forecasting, supply chain optimisation,
quality control, and predictive maintenance.

### Demand Forecasting

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

np.random.seed(0)
dates      = pd.date_range("2020-01-01", periods=104, freq="W")  # 2 years of weekly data
trend      = np.linspace(500, 800, 104)
seasonality= 100 * np.sin(2 * np.pi * np.arange(104) / 52)
noise      = np.random.normal(0, 30, 104)
demand     = (trend + seasonality + noise).clip(0).round()

df_ops = pd.DataFrame({
    "Date"         : dates,
    "Demand"       : demand,
    "Week_of_Year" : dates.isocalendar().week.values,
    "Quarter"      : dates.quarter,
    "Promotion"    : np.random.choice([0, 1], 104, p=[0.75, 0.25]),
    "Price_Index"  : np.random.uniform(0.95, 1.05, 104).round(3),
})

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_ops["Date"], df_ops["Demand"], color="steelblue", linewidth=1.2)
ax.set_title("Weekly Product Demand (2 Years)")
ax.set_xlabel("Date")
ax.set_ylabel("Units Demanded")
plt.tight_layout()
plt.show()

In [ ]:
# Feature engineering for forecasting
X = df_ops[["Week_of_Year", "Quarter", "Promotion", "Price_Index"]]
y = df_ops["Demand"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, shuffle=False)      # no shuffle — time order matters!

rf_ops = RandomForestRegressor(n_estimators=200, random_state=42)
rf_ops.fit(X_tr, y_tr)
y_pred_ops = rf_ops.predict(X_te)

mae = mean_absolute_error(y_te, y_pred_ops)
r2  = r2_score(y_te, y_pred_ops)
print(f"Demand Forecast MAE : {mae:.1f} units")
print(f"Demand Forecast R²  : {r2:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_te.values, label="Actual Demand", color="steelblue")
ax.plot(y_pred_ops,  label="Forecast",      color="orange", linestyle="--")
ax.set_title("Demand Forecast vs Actual (Test Period)")
ax.set_xlabel("Week")
ax.set_ylabel("Units")
ax.legend()
plt.tight_layout()
plt.show()

### Predictive Maintenance

In [ ]:
np.random.seed(22)
n = 1000

# Machine sensor readings
df_maint = pd.DataFrame({
    "Temperature"   : np.random.normal(75, 10, n),
    "Vibration"     : np.random.normal(0.5, 0.1, n),
    "Operating_Hrs" : np.random.randint(100, 10_000, n),
    "Pressure"      : np.random.normal(100, 15, n),
})

# Failure more likely when temperature is high and vibration is high
failure_prob = (
    0.001 * df_maint["Temperature"] +
    0.3   * df_maint["Vibration"]   +
    0.00001 * df_maint["Operating_Hrs"] - 0.05
).clip(0, 1)

df_maint["Failure"] = np.random.binomial(1, failure_prob)
print(f"Failure rate: {df_maint['Failure'].mean():.2%}")

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_m = df_maint.drop(columns="Failure")
y_m = df_maint["Failure"]
X_mtr, X_mte, y_mtr, y_mte = train_test_split(X_m, y_m, test_size=0.2,
                                                random_state=42, stratify=y_m)

rf_maint = RandomForestClassifier(n_estimators=100, random_state=42)
rf_maint.fit(X_mtr, y_mtr)
print(classification_report(y_mte, rf_maint.predict(X_mte),
                             target_names=["OK","Failure"]))

---

### Student Task 4.3

1. Using `df_ops`, investigate whether **promotions** significantly increase demand. Calculate average demand with and without a promotion.
2. Retrain the demand forecasting model adding a **Lag_1_Demand** feature (last week's demand). Does R² improve?
3. For the predictive maintenance model, plot feature importance. Which sensor reading is the strongest predictor of machine failure?
4. Describe how a manufacturer could use this model to **reduce downtime costs**.

In [ ]:
# Your code here

---

### Evaluation Questions 4.3

1. Why should demand forecasting data **not be shuffled** before splitting train/test?
   a) Shuffling causes data loss
   b) The time sequence must be preserved so the model does not see future data (correct)
   c) Shuffling makes models slower to train
   d) Demand data is already sorted by default

2. A **lag feature** (e.g., last week's demand) is useful because:
   a) It reduces the training set size
   b) It captures temporal patterns and autocorrelation (correct)
   c) It replaces the need for trend features
   d) It removes seasonality from the data

3. **Predictive maintenance** uses ML to:
   a) Automate equipment purchasing
   b) Identify which machines are most expensive
   c) Predict equipment failure before it occurs to schedule proactive maintenance (correct)
   d) Optimise the number of shifts per day

4. What is the business benefit of reducing **false negatives** in a machine failure model?
   a) Fewer unnecessary maintenance interventions
   b) Lower probability of unexpected breakdowns and costly downtime (correct)
   c) Better accuracy on the training set
   d) Reduced energy consumption

5. Which ML model type is most appropriate for **predicting a continuous quantity** like weekly demand?
   a) Logistic Regression
   b) K-Means Clustering
   c) Random Forest Regressor (correct)
   d) Isolation Forest

---